In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from hatua_ml.data.load_kdhs import load_kdhs
from hatua_ml.features.ecdi2030 import ITEMS

df = load_kdhs()
complete = df.dropna(subset=ITEMS + ["on_track"])
X = complete[ITEMS + ["age_months"]].astype(float)
y = (1 - complete["on_track"]).astype(int)      # 1 = off track (the risk class)

print(f"loaded {len(df)}, complete cases {len(complete)}, off-track rate {y.mean():.1%}")

loaded 5478, complete cases 5195, off-track rate 29.4%


In [2]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": "accuracy",
    "sensitivity": "recall",
    "specificity": make_scorer(recall_score, pos_label=0),
    "auc": "roc_auc",
}
models = {
    "logreg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "xgboost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                             eval_metric="logloss", random_state=42),
}

rows = []
for name, model in models.items():
    s = cross_validate(model, X, y, cv=cv, scoring=scoring)
    rows.append({"model": name, **{k: s[f"test_{k}"].mean() for k in scoring}})
pd.DataFrame(rows).round(3)

,model,accuracy,sensitivity,specificity,auc
0,logreg,0.952,0.912,0.969,0.991
1,xgboost,0.979,0.954,0.989,0.998


In [3]:
from sklearn.tree import DecisionTreeClassifier, export_text

X_rule = pd.DataFrame({
    "items_passed": complete[ITEMS].astype(float).sum(axis=1),
    "age_months": complete["age_months"],
})
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
s = cross_validate(tree, X_rule, y, cv=cv, scoring=scoring)
print({k: round(s[f"test_{k}"].mean(), 3) for k in scoring})

tree.fit(X_rule, y)
print(export_text(tree, feature_names=list(X_rule.columns)))

{'accuracy': np.float64(0.912), 'sensitivity': np.float64(0.885), 'specificity': np.float64(0.923), 'auc': np.float64(0.968)}
|--- items_passed <= 14.50
|   |--- age_months <= 47.50
|   |   |--- items_passed <= 8.50
|   |   |   |--- age_months <= 29.50
|   |   |   |   |--- class: 1
|   |   |   |--- age_months >  29.50
|   |   |   |   |--- class: 1
|   |   |--- items_passed >  8.50
|   |   |   |--- age_months <= 41.50
|   |   |   |   |--- class: 0
|   |   |   |--- age_months >  41.50
|   |   |   |   |--- class: 1
|   |--- age_months >  47.50
|   |   |--- class: 1
|--- items_passed >  14.50
|   |--- class: 0

